In [ ]:
!pip install --upgrade protobuf

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import random

from datasets import Dataset, DatasetDict, ClassLabel
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEED = 2025
np.random.seed(SEED)
plt.rc('font', family='NanumGothic')
plt.style.use('ggplot')
print("  - 딥러닝 라이브러리 (swifter 포함) 임포트 완료.")

In [ ]:
df = pd.read_csv('./data/munch_labeled_RESUMED_200125.csv')

In [ ]:
df

In [ ]:
# 2. 학습 데이터 로드 (Gemini가 라벨링한 데이터)
# 파일명은 사용자가 생성한 최종 파일명으로 지정합니다.
DATA_PATH = "./data/munch_labeled_RESUMED_200125.csv" 

try:
    print(f"\n📂 [Step 2] 데이터 로드 중... ({DATA_PATH})")
    df = pd.read_csv(DATA_PATH)
    
    # 데이터 확인
    print(f"  - 전체 데이터 개수: {len(df)}")
    print(f"  - 컬럼 목록: {df.columns.tolist()}")
    
    # 3. 데이터 전처리 (Preprocessing)
    print("\n🛠️ [Step 3] 데이터 전처리 (BERT 입력 형식 변환)")
    
    # 필요한 컬럼만 선택 및 이름 변경 (HuggingFace Trainer 포맷 준수)
    # original -> text (입력)
    # difficulty_label -> label (정답)
    df_train = df[['original', 'difficulty_label']].copy()
    df_train.rename(columns={'original': 'text', 'difficulty_label': 'label'}, inplace=True)
    
    # 결측치 제거 (Text가 비어있거나 Label이 없는 경우)
    df_train.dropna(subset=['text', 'label'], inplace=True)
    
    # 라벨 정수형 변환 (1.0 -> 1)
    df_train['label'] = df_train['label'].astype(int)
    
    # 데이터 분포 확인
    print(f"  - 정제 후 데이터 개수: {len(df_train)}")
    print("  - 라벨 분포 (0: Easy, 1: Hard):")
    print(df_train['label'].value_counts())
    
    # Pandas DataFrame -> Hugging Face Dataset 변환
    dataset = Dataset.from_pandas(df_train)
    
    # 라벨 타입 캐스팅 (ClassLabel)
    dataset = dataset.cast_column("label", ClassLabel(num_classes=2, names=['easy', 'hard']))
    
    # Train/Test 분리 (8:2 비율, 층화 추출)
    train_test_split = dataset.train_test_split(test_size=0.2, seed=SEED, stratify_by_column='label')
    dataset_dict = DatasetDict({
        'train': train_test_split['train'],
        'test': train_test_split['test']
    })
    
    print("\n✅ 데이터셋 준비 완료:")
    print(dataset_dict)

except FileNotFoundError:
    print(f"❌ 오류: {DATA_PATH} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# 4. 모델 및 토크나이저 로드
# 한국어 성능이 우수한 kykim/bert-kor-base 사용 (혹은 klue/roberta-base 추천)
MODEL_NAME = "kykim/bert-kor-base" 
print(f"\n🤖 [Step 4] 모델 로드: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# 토크나이징 함수 (Dynamic Padding 사용 예정이므로 max_length만 지정하고 padding=False)
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # 배치 단위 동적 패딩

# 5. 학습 설정 (Training Arguments)
print("\n⚙️ [Step 5] 학습 파라미터 설정")

# 저장 경로 설정
OUTPUT_DIR = "./output/smart_router_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,              # 5 에폭 학습
    learning_rate=2e-5,              # 학습률 (BERT 계열 표준)
    
    # [배치 사이즈 설정]
    # 10GB VRAM + FP16 기준: Train 32 / Eval 64 정도가 한계 효율입니다.
    # 만약 OOM(Out Of Memory) 에러가 나면 train을 16으로 줄이세요.
    per_device_train_batch_size=32,  
    per_device_eval_batch_size=64,   # 평가는 역전파가 없어 메모리를 덜 먹으므로 더 크게 설정
    
    warmup_ratio=0.1,                # 학습 초반 안정화
    weight_decay=0.01,
    
    eval_strategy="epoch",           # 매 에폭마다 평가
    save_strategy="epoch",           # 매 에폭마다 모델 저장
    load_best_model_at_end=True,     # 가장 성능 좋은 모델 로드
    metric_for_best_model="f1",      # 평가 기준은 F1 Score
    
    report_to="none",                # WandB 등 리포팅 끄기
    seed=SEED,
    
    # [속도 및 메모리 최적화]
    # A100은 FP16 연산에 특화되어 있어 속도가 빨라지고 메모리는 절반으로 줍니다.
    # 10GB 메모리에서 배치사이즈를 키우기 위해 필수입니다.
    fp16=True,                       
    
    # 데이터 로딩 속도 향상 (CPU 코어 활용)
    dataloader_num_workers=4,
    
    # 입력 문장 길이를 비슷한 것끼리 묶어 패딩을 줄임 (학습 속도 향상)
    group_by_length=True,
)

# 평가 지표 함수
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro') # 클래스 불균형 고려 Macro F1
    return {"accuracy": acc, "f1": f1}

# Trainer 초기화
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # 성능 개선 없으면 조기 종료
)

# 6. 학습 시작
trainer.train()

# 7. 최종 평가 및 시각화
print("\n📊 [Step 7] 최종 성능 평가")
eval_results = trainer.evaluate()
print(f"  - Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"  - F1 Score: {eval_results['eval_f1']:.4f}")

# 혼동 행렬 (Confusion Matrix) 시각화
print("\n🎨 혼동 행렬 시각화 생성 중...")
predictions = trainer.predict(tokenized_datasets["test"])
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

cm = confusion_matrix(y_true, y_pred)

# 시각화 (0: Easy, 1: Hard)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Pred: Easy(0)', 'Pred: Hard(1)'],
            yticklabels=['True: Easy(0)', 'True: Hard(1)'])
plt.title('Smart Router Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')

# 이미지 저장
save_img_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')
plt.savefig(save_img_path)
print(f"  - 혼동 행렬 이미지 저장됨: {save_img_path}")

# 모델 저장
trainer.save_model(OUTPUT_DIR)
print(f"\n💾 [Complete] 모델 저장 완료: {OUTPUT_DIR}")
print("이제 이 모델을 사용하여 '스마트 라우터'를 가동할 수 있습니다.")